### Available Data Sources
For this analysis, we will use data from the [Pacific Northwest National Laboratory's Data Center Atlas](https://im3.pnnl.gov/datacenter-atlas) (PNNL) and the [FracTracker Alliance's National Data Centers Tracker](https://www.fractracker.org/2025/07/national-data-centers-tracker/) (FTA).

The PNNL dataset contains locations of existing data center facilities in the United States. Data center locations were derived from OpenStreetMap (OSM), a crowd-sourced database. Data points from OSM are processed in various ways to determine additional variables provided in the data including: facility area (square feet), associated US county, and US state.

The FTA data set maps existing, permitted, and proposed data-center facilities across the U.S., built to analyze their environmental, regulatory, and energy impacts.

##### Variables

Original PNNL:
- **id** - unique identification number (OSM-provided with prefix of "node/", "relation/" and similar attributes removed).
- **state**
- **state_abb**
- **state_id**
- **county**
- **county_id**
- **ref** - reference numbers or codes (OSM-provided).
- **operator** - the name of the company, corporation, or person in charge facility (OSM-provided).
- **name** - name of facility (OSM-provided) 
- **sqft** - surface area of facility polygon, measured in square feet. Only available for "building" and "campus" layers. 
- **lat** - latitude of data centroid point 
- **lon** - longitude of data centroid point 
- **type** – represented spatial information. One of "point", "building", or "campus". 

Original FTA
- **facility_name**: Name of the data center facility or project. This may refer to the official facility name, development project name, or commonly used name in public records or media reports.
- **address**: Street address of the data center facility when available.
- **city**: City or municipality where the facility is located.
- **state**: U.S. state in which the facility is located.
- **zip**: ZIP code corresponding to the facility location.
- **county**: County in which the facility is located.
- **lat**: Latitude coordinate representing the geographic location of the facility.
- **lon**: Longitude coordinate representing the geographic location of the facility.
- **status**: Current stage of the data center project (e.g., existing/operational, under construction, permitted, or proposed).
- **location_confidence**: Indicator of how precise the geographic coordinates are, reflecting whether the location represents an exact facility site or an approximate location.
- **purpose**: Primary function or intended use of the facility, such as cloud computing, artificial intelligence workloads, hyperscale data processing, or colocation services.
- **operator_name**: Company responsible for operating or managing the data center facility.
- **tenant**: Major tenant(s) or companies leasing computing capacity within the facility.
- **mw**: Estimated or reported electrical capacity of the data center, measured in megawatts (MW).
- **sizerank**: Relative ranking or categorical classification of the facility’s size within the dataset, typically based on power capacity or facility scale.
- **power_source**: Primary source of electricity used to power the data center (e.g., grid electricity, natural gas, renewable energy, or mixed sources).
- **dedicated_power_plant**: Indicator of whether the data center is supported by a dedicated power generation facility built specifically to supply its electricity demand.
- **number_of_generators**: Number of backup generators installed or planned at the facility, typically used to provide power during grid outages.
- **number_of_buildings**: Number of buildings or structures comprising the data center campus.
- **cooling_source**: Source of water or other medium used for cooling the facility (e.g., municipal water supply, groundwater, surface water, or recycled water).
- **cooling_type**: Type of cooling technology used at the facility, such as air cooling, evaporative cooling, liquid cooling, or hybrid systems.
- **facility_size_sqft**: Total building floor area of the data center facility measured in square feet.
- **property_size_acres**: Total land area of the data center site measured in acres.
- **project_cost**: Estimated capital cost of constructing the data center project.
- **expected_date_online**: Projected date when the facility is expected to begin operations.
- **community_pushback**: Indicator or notes describing whether local residents or community groups have expressed opposition to the project.
- **advocacy_information**: Additional information about advocacy efforts, community organizing, or campaigns related to the facility.
- **resistance_status**: Categorization of the level or stage of community resistance (e.g., none, emerging opposition, organized campaign, or legal challenges).
- **nda**: Indicates whether a non-disclosure agreement (NDA) is associated with the project, potentially limiting publicly available information about the facility.


### Join Datasets

Join the PNNL and FTA datasets together. Using fuzzy spatial matching, combine both data sets using `lon` and `lat`. We will use the PNNL dataset as the base that the FTA dataset is matched and joined to.

In [ ]:
# Import packages
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np


In [5]:
# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path.cwd().parents[1]

IM3_FILE = PROJECT_ROOT / "data" / "raw" / "im3" / "im3_open_source_data_center_atlas_v2026.02.09.csv"
FRACTRACKER_FILE = PROJECT_ROOT / "data" / "raw" / "fta" / "fractracker_database.xlsx"

OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "preprocessing" / "datacenter_data" / "datacenters_matched.xlsx"

In [6]:
# -----------------------------
# Parameters
# -----------------------------
MAX_DISTANCE_M = 2000

In [7]:
# -----------------------------
# Load data
# -----------------------------
im3 = pd.read_csv(IM3_FILE)
fractracker = pd.read_excel(FRACTRACKER_FILE)

# -----------------------------
# Filter to Virginia
# -----------------------------
im3_va = im3[
    im3["state_abb"].astype(str).str.strip().str.upper().eq("VA")
    | im3["state"].astype(str).str.strip().str.upper().eq("VIRGINIA")
].copy()

fractracker_va = fractracker[
    fractracker["state"].astype(str).str.strip().str.upper().isin(["VA", "VIRGINIA"])
].copy()

In [8]:
# -----------------------------
# Clean coordinates
# -----------------------------
for df in (im3_va, fractracker_va):
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

im3_va = im3_va.dropna(subset=["lat", "lon"]).copy()
fractracker_va = fractracker_va.dropna(subset=["lat", "lon"]).copy()

# Basic coordinate sanity check
im3_va = im3_va[(im3_va["lat"].between(20, 50)) & (im3_va["lon"].between(-130, -60))].copy()
fractracker_va = fractracker_va[
    (fractracker_va["lat"].between(20, 50)) & (fractracker_va["lon"].between(-130, -60))
].copy()

# -----------------------------
# Add IDs so matches are traceable
# -----------------------------
im3_va = im3_va.reset_index(drop=True)
fractracker_va = fractracker_va.reset_index(drop=True)

im3_va["im3_id"] = im3_va.index + 1
fractracker_va["fractracker_id"] = fractracker_va.index + 1

# Rename fractracker columns before join so output is readable
fractracker_va = fractracker_va.rename(
    columns={
        "facility_name": "fractracker_facility_name",
        "address": "fractracker_address",
        "city": "fractracker_city",
        "county": "fractracker_county",
        "operator_name": "fractracker_operator_name",
        "status": "fractracker_status",
        "purpose": "fractracker_purpose",
        "mw": "fractracker_mw",
        "location_confidence": "fractracker_location_confidence",
        "lat": "fractracker_lat",
        "lon": "fractracker_lon",
    }
)

# -----------------------------
# Convert to GeoDataFrames
# -----------------------------
im3_gdf = gpd.GeoDataFrame(
    im3_va,
    geometry=gpd.points_from_xy(im3_va["lon"], im3_va["lat"]),
    crs="EPSG:4326",
)

fractracker_gdf = gpd.GeoDataFrame(
    fractracker_va,
    geometry=gpd.points_from_xy(fractracker_va["fractracker_lon"], fractracker_va["fractracker_lat"]),
    crs="EPSG:4326",
)

# Reproject so distance is in meters
# EPSG:3857 is acceptable for this matching task
im3_proj = im3_gdf.to_crs(epsg=3857)
fractracker_proj = fractracker_gdf.to_crs(epsg=3857)

# -----------------------------
# Nearest-neighbor fuzzy spatial match
# Keep all im3 rows
# -----------------------------
matched = gpd.sjoin_nearest(
    im3_proj,
    fractracker_proj,
    how="left",
    max_distance=MAX_DISTANCE_M,
    distance_col="match_distance_m",
    lsuffix="im3",
    rsuffix="fractracker",
)

# Drop geometry and index artifacts
matched = matched.drop(columns=["geometry", "index_fractracker"], errors="ignore")

# Flag matched rows
matched["matched_flag"] = matched["fractracker_id"].notna().astype(int)

# Match quality buckets
def classify_match(distance):
    if pd.isna(distance):
        return "no match"
    if distance <= 500:
        return "strong"
    if distance <= 2000:
        return "plausible"
    return "weak"

matched["match_quality"] = matched["match_distance_m"].apply(classify_match)

# -----------------------------
# Optional name similarity check
# -----------------------------
from difflib import SequenceMatcher

def name_similarity(a, b):
    if pd.isna(a) or pd.isna(b):
        return None
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

matched["name_similarity"] = matched.apply(
    lambda row: name_similarity(row.get("name"), row.get("fractracker_facility_name")),
    axis=1,
)

# -----------------------------
# Save & Remove unwanted columns
# -----------------------------

# Convert facility_size_sqft from string to numeric
matched["facility_size_sqft"] = (
    matched["facility_size_sqft"]
    .astype(str)
    .str.replace(",", "", regex=False)   # remove thousand separators
    .str.strip()
)

matched["facility_size_sqft"] = pd.to_numeric(
    matched["facility_size_sqft"],
    errors="coerce"
)

# Columns from Fractracker we do NOT want in the final dataset
drop_columns = [
    "state_fractracker",
    "fractracker_purpose",
    "tenant",
    "power_source",
    "dedicated_power_plant",
    "number_of_generators",
    "number_of_buildings",
    "cooling_source",
    "cooling_type",
    "project_cost",
    "expected_date_online",
    "community_pushback",
    "advocacy_information",
    "resistance_status",
    "nda",
    "community_group_website_1",
    "community_group_website_2",
    "petition_url",
    "info_source_1",
    "info_source_2",
    "info_source_3",
    "info_source_4",
    "info_source_5",
    "info_source_6",
    "info_source_7",
    "info_source_8",
    "date_created",
    "date_updated"
]


# remove unwanted fractracker variables
matched = matched.drop(columns=drop_columns, errors="ignore")

matched.to_excel(OUTPUT_FILE, index=False)

# -----------------------------
# Print summary
# -----------------------------
print(f"im3 Virginia rows: {len(im3_va)}")
print(f"fractracker Virginia rows: {len(fractracker_va)}")
print(f"Matched im3 rows: {matched['matched_flag'].sum()}")
print(f"Unmatched im3 rows: {(matched['matched_flag'] == 0).sum()}")
print(f"Output written to: {OUTPUT_FILE}")

im3 Virginia rows: 319
fractracker Virginia rows: 452
Matched im3 rows: 316
Unmatched im3 rows: 3
Output written to: c:\Corbin_Documents\Hertie, Class of 2026\Course Work\Theses\2025-26\Master-Thesis-2026\data\processed\preprocessing\datacenter_data\datacenters_matched.xlsx


### Years of Operation

Using the PNNL dataset as a basis, we collected the beginning year of operation for eact data center when it was available. If the beginning date of operation was not available, we also collected year construction completion and the announcement year.

In [9]:
PNNL_YRS_FILE = PROJECT_ROOT / "data" / "raw" / "im3_years" / "va_pnnl_researched.xlsx"

pnnl_yrs = pd.read_excel(PNNL_YRS_FILE)

pnnl_yrs.columns = pnnl_yrs.columns.str.strip().str.lower().str.replace(" ", "_")

In [10]:
# Subset to relevant columns for merging with main dataset

pnnl_yrs_subset = pnnl_yrs[
    ["id", "research_year", "year_basis_used", "year_source_link"]
].copy()

In [11]:
matched = matched.merge(
    pnnl_yrs_subset,
    on="id",
    how="left"
)

matched.to_excel(OUTPUT_FILE, index=False)

### Append to census tract

In [12]:
TRACT_FILE = PROJECT_ROOT / "data" / "raw" / "census_data" / "tl_2016_51_tract"

# 1. Read census tract polygons from zip
tracts = gpd.read_file(TRACT_FILE)

# 2. Check tract columns
print(tracts.columns.tolist())

# 3. Convert your output dataset to GeoDataFrame using im3 lon/lat
matched_gdf = gpd.GeoDataFrame(
    matched.copy(),
    geometry=gpd.points_from_xy(matched["lon"], matched["lat"]),
    crs="EPSG:4326"
)

# 4. Make sure both layers use the same CRS
tracts = tracts.to_crs(matched_gdf.crs)

# 5. Keep only the tract columns you want
tracts_keep = tracts[["GEOID", "NAME", "STATEFP", "COUNTYFP", "TRACTCE", "geometry"]].copy()

# 6. Spatial join: assign each point to the tract polygon it falls within
matched_with_tract = gpd.sjoin(
    matched_gdf,
    tracts_keep,
    how="left",
    predicate="within"
)

# 7. Clean up join artifacts
matched_with_tract = matched_with_tract.drop(columns=["geometry", "index_right"], errors="ignore")

# 8. Optional: rename tract variables for clarity
matched_with_tract = matched_with_tract.rename(columns={
    "GEOID": "census_tract_geoid",
    "NAME": "census_tract_name",
    "STATEFP": "census_statefp",
    "COUNTYFP": "census_countyfp",
    "TRACTCE": "census_tractce"
})

print(matched_with_tract[[
    "id", "census_tract_geoid", "census_tract_name"
]].head())

['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']
         id census_tract_geoid census_tract_name
0  14997588        51087201404           2014.04
1  19787207        51153901504           9015.04
2  45583050        51153901503           9015.03
3  45776978        51153901408           9014.08
4  45776979        51153901408           9014.08


In [13]:
final_output = matched_with_tract.copy()

OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "for_analysis" / "datacenters_matched_tracts.xlsx"
final_output.to_excel(OUTPUT_FILE, index=False)
print("Saved to:", OUTPUT_FILE)

Saved to: c:\Corbin_Documents\Hertie, Class of 2026\Course Work\Theses\2025-26\Master-Thesis-2026\data\processed\for_analysis\datacenters_matched_tracts.xlsx
